# Government Timeline — Build Process

Builds `government_timeline.json`: one row per continuous PM tenure under a
single electoral mandate. Entries are split where a PM won re-election, or
where a government changed composition mid-parliament without an election.

**This notebook does not run cleanly top to bottom.** Two points require
manual intervention outside the code:

1. After the cell that exports `pm_timeline_working.xlsx` and
   `election_data.xlsx`, the timeline file was filled in by hand in Excel —
   election dates, own_mandate, opposition_party, coalition_parties — using
   the sources listed in the folder README. That edited file is what the
   later cells read back in.
2. The final cell merges in `end_reason` and `notes` from `text.txt`, which
   was generated separately using Gemini 3.1 Pro, not written here.

See the folder README for the full source list and file inventory.

## Election Data

In [1]:
import pandas as pd

df = pd.read_excel('Book1.ods', engine='odf')
print(df.head())

ImportError: `Import odfpy` failed.  Use pip or conda to install the odfpy package.

In [10]:
# Drop the sub-header row
df = df.drop(index=0).reset_index(drop=True)

# Rename columns cleanly
df.columns = [
    'election_date', 'total_seats',
    'con_votes', 'con_seats',
    'lab_votes', 'lab_seats',
    'lib_votes', 'lib_seats',
    'oth_votes', 'oth_seats',
    'comment'
]

print(df.head())

         election_date total_seats con_votes con_seats lab_votes lab_seats  \
0  1910-12-19 00:00:00         670      46.3       272       7.2        42   
1  1918-12-14 00:00:00         707      38.7       383      23.7        73   
2  1922-11-15 00:00:00         615      38.2       345      29.5       142   
3  1923-12-06 00:00:00         615      38.1       258      30.5       191   
4  1924-10-29 00:00:00         615      48.3       419        33       151   

  lib_votes lib_seats oth_votes oth_seats  \
0      43.8       272       2.7        84   
1      25.6       161        12        90   
2      29.1       116       3.2        12   
3      29.6       159       1.8         7   
4      17.6        40       1.1         5   

                                             comment  
0  Repeat election forced by George V to legitimi...  
1        Lloyd George (Lib) leads coalition with Con  
2  Bonar Law (Con) breaks the coalition, Labour o...  
3  Ramsay MacDonald (Lab) with Lib-coali

In [ ]:
df['election_date'] = pd.to_datetime(df['election_date']).dt.strftime('%Y-%m-%d')

numeric_cols = ['total_seats', 'con_votes', 'con_seats', 'lab_votes', 'lab_seats', 
                'lib_votes', 'lib_seats', 'oth_votes', 'oth_seats']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [35]:
election_df = df.copy()

## Timeline table

In [14]:
import pandas as pd

columns = [
    'pm',
    'party',
    'election_date',
    'own_mandate',
    'seats_won',
    'total_seats',
    'majority',
    'majority_end',
    'opposition_party',
    'coalition_parties',
    'pm_end_date',
    'resignation_date',
    'end_reason',
    'resignation',
    'notes'
]

df_timeline = pd.DataFrame(columns=columns)
print(df_timeline)

Empty DataFrame
Columns: [pm, party, election_date, own_mandate, seats_won, total_seats, majority, majority_end, opposition_party, coalition_parties, pm_end_date, resignation_date, end_reason, resignation, notes]
Index: []


## Primemister data

In [29]:
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_prime_ministers_of_the_United_Kingdom"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

html = StringIO(response.text)

tables = pd.read_html(html)

print(len(tables))
print(tables[0].head())

4
  Portrait                                Prime ministerOffice(lifespan)  \
  Portrait Portrait.1                     Prime ministerOffice(lifespan)   
0        ​        NaN  Robert Walpole[26]MP for King's Lynn (to 1742)...   
1        ​        NaN  Robert Walpole[26]MP for King's Lynn (to 1742)...   
2        ​        NaN  Robert Walpole[26]MP for King's Lynn (to 1742)...   
3        ​        NaN  Robert Walpole[26]MP for King's Lynn (to 1742)...   
4        ​        NaN  Spencer Compton[27]1st Earl of Wilmington(1673...   

     Term of office                                        Mandate[a]  \
              Start               End             Duration Mandate[a]   
0      3 April 1721  11 February 1742   20 years, 315 days       1722   
1      3 April 1721  11 February 1742   20 years, 315 days       1727   
2      3 April 1721  11 February 1742   20 years, 315 days       1734   
3      3 April 1721  11 February 1742   20 years, 315 days       1741   
4  16 February 1742       2

In [30]:
for i, t in enumerate(tables):
    print(f"\n=== Table {i} ===")
    print(t.shape)
    print(t.iloc[0])


=== Table 0 ===
(126, 11)
Portrait                                    Portrait                                                                                      ​
                                            Portrait.1                                                                                  NaN
Prime ministerOffice(lifespan)              Prime ministerOffice(lifespan)                Robert Walpole[26]MP for King's Lynn (to 1742)...
Term of office                              Start                                                                              3 April 1721
                                            End                                                                            11 February 1742
                                            Duration                                                                     20 years, 315 days
Mandate[a]                                  Mandate[a]                                                                               

In [31]:
df_pm = tables[0].copy()

# Extract just the columns we need using tuple keys
df_pm = df_pm[[
    ('Prime ministerOffice(lifespan)', 'Prime ministerOffice(lifespan)'),
    ('Term of office', 'Start'),
    ('Term of office', 'End'),
    ('Party', 'Party')
]]

# Flatten column names
df_pm.columns = ['pm_raw', 'start', 'end', 'party']

# Parse dates
df_pm['start'] = pd.to_datetime(df_pm['start'], dayfirst=True, errors='coerce')
df_pm['end'] = pd.to_datetime(df_pm['end'], dayfirst=True, errors='coerce')

# Filter to 1910 onwards
df_pm = df_pm[df_pm['start'] >= '1910-01-01'].reset_index(drop=True)

print(df_pm)

                                               pm_raw      start        end  \
0   David Lloyd George[73]MP for Carnarvon Borough... 1916-12-06 1922-10-19   
1   David Lloyd George[73]MP for Carnarvon Borough... 1916-12-06 1922-10-19   
2      Bonar Law[74]MP for Glasgow Central(1858–1923) 1922-10-23 1923-05-20   
3        Stanley Baldwin[75]MP for Bewdley(1867–1947) 1923-05-22 1924-01-22   
4      Ramsay MacDonald[76]MP for Aberavon(1866–1937) 1924-01-22 1924-11-04   
5        Stanley Baldwin[77]MP for Bewdley(1867–1947) 1924-11-04 1929-06-04   
6        Ramsay MacDonald[78]MP for Seaham(1866–1937) 1929-06-05 1935-06-07   
7        Ramsay MacDonald[78]MP for Seaham(1866–1937) 1929-06-05 1935-06-07   
8        Ramsay MacDonald[78]MP for Seaham(1866–1937) 1929-06-05 1935-06-07   
9        Stanley Baldwin[79]MP for Bewdley(1867–1947) 1935-06-07 1937-05-28   
10       Stanley Baldwin[79]MP for Bewdley(1867–1947) 1935-06-07 1937-05-28   
11       Stanley Baldwin[79]MP for Bewdley(1867–1947

In [32]:
# Clean PM name
df_pm['pm'] = df_pm['pm_raw'].str.extract(r'^([^\[]+)').iloc[:, 0].str.strip()

# Clean party
df_pm['party'] = df_pm['party'].str.replace(r'\(Scot\.U\.\)', '', regex=True).str.strip()
df_pm['party'] = df_pm['party'].str.extract(r'^([^\(]+)').iloc[:, 0].str.strip()

# Deduplicate
df_pm = df_pm.drop_duplicates(subset=['start', 'end']).reset_index(drop=True)

# Format dates
df_pm['start'] = df_pm['start'].dt.strftime('%Y-%m-%d')
df_pm['end'] = df_pm['end'].dt.strftime('%Y-%m-%d')

# Keep only what we need
df_pm = df_pm[['pm', 'party', 'start', 'end']]

print(df_pm.to_string())

                     pm              party       start         end
0    David Lloyd George  Coalition Liberal  1916-12-06  1922-10-19
1             Bonar Law       Conservative  1922-10-23  1923-05-20
2       Stanley Baldwin       Conservative  1923-05-22  1924-01-22
3      Ramsay MacDonald             Labour  1924-01-22  1924-11-04
4       Stanley Baldwin       Conservative  1924-11-04  1929-06-04
5      Ramsay MacDonald             Labour  1929-06-05  1935-06-07
6       Stanley Baldwin       Conservative  1935-06-07  1937-05-28
7   Neville Chamberlain       Conservative  1937-05-28  1940-05-10
8     Winston Churchill       Conservative  1940-05-10  1945-07-26
9        Clement Attlee             Labour  1945-07-26  1951-10-26
10    Winston Churchill       Conservative  1951-10-26  1955-04-05
11         Anthony Eden       Conservative  1955-04-06  1957-01-09
12     Harold Macmillan       Conservative  1957-01-10  1963-10-18
13    Alec Douglas-Home       Conservative  1963-10-19  1964-1

In [33]:
# Add Asquith
asquith = pd.DataFrame([{
    'pm': 'H. H. Asquith',
    'party': 'Liberal',
    'start': '1910-12-19',
    'end': '1916-12-06'
}])

# Split MacDonald 1929-1935 into two entries
# Remove the single 1929-1935 entry
df_pm = df_pm[~((df_pm['pm'] == 'Ramsay MacDonald') & (df_pm['start'] == '1929-06-05'))]

# Add the two correct entries
macdonald_labour = pd.DataFrame([{
    'pm': 'Ramsay MacDonald',
    'party': 'Labour',
    'start': '1929-06-05',
    'end': '1931-08-24'
}])

macdonald_national = pd.DataFrame([{
    'pm': 'Ramsay MacDonald',
    'party': 'National Labour',
    'start': '1931-08-24',
    'end': '1935-06-07'
}])

# Combine and sort
df_pm = pd.concat([asquith, df_pm, macdonald_labour, macdonald_national], ignore_index=True)
df_pm = df_pm.sort_values('start').reset_index(drop=True)

print(df_pm.to_string())

                     pm              party       start         end
0         H. H. Asquith            Liberal  1910-12-19  1916-12-06
1    David Lloyd George  Coalition Liberal  1916-12-06  1922-10-19
2             Bonar Law       Conservative  1922-10-23  1923-05-20
3       Stanley Baldwin       Conservative  1923-05-22  1924-01-22
4      Ramsay MacDonald             Labour  1924-01-22  1924-11-04
5       Stanley Baldwin       Conservative  1924-11-04  1929-06-04
6      Ramsay MacDonald             Labour  1929-06-05  1931-08-24
7      Ramsay MacDonald    National Labour  1931-08-24  1935-06-07
8       Stanley Baldwin       Conservative  1935-06-07  1937-05-28
9   Neville Chamberlain       Conservative  1937-05-28  1940-05-10
10    Winston Churchill       Conservative  1940-05-10  1945-07-26
11       Clement Attlee             Labour  1945-07-26  1951-10-26
12    Winston Churchill       Conservative  1951-10-26  1955-04-05
13         Anthony Eden       Conservative  1955-04-06  1957-0

## Building the timeline table

In [48]:
print(election_df.columns)
print(df_pm.columns)
print(df_timeline.columns)

Index(['election_date', 'total_seats', 'con_votes', 'con_seats', 'lab_votes',
       'lab_seats', 'lib_votes', 'lib_seats', 'oth_votes', 'oth_seats',
       'comment'],
      dtype='str')
Index(['pm', 'party', 'start', 'end'], dtype='str')
Index(['pm', 'party', 'pm_start_date', 'pm_end_date', 'election_date',
       'own_mandate', 'seats_won', 'total_seats', 'majority', 'majority_end',
       'opposition_party', 'coalition_parties', 'resignation_date',
       'end_reason', 'resignation', 'notes'],
      dtype='str')


In [54]:
# Start fresh from df_pm
df_timeline = df_pm.copy().rename(columns={'start': 'pm_start_date', 'end': 'pm_end_date'})
print(display(df_timeline))

,pm,party,pm_start_date,pm_end_date
0,H. H. Asquith,Liberal,1910-12-19,1916-12-06
1,David Lloyd George,Coalition Liberal,1916-12-06,1922-10-19
2,Bonar Law,Conservative,1922-10-23,1923-05-20
3,Stanley Baldwin,Conservative,1923-05-22,1924-01-22
4,Ramsay MacDonald,Labour,1924-01-22,1924-11-04
5,Stanley Baldwin,Conservative,1924-11-04,1929-06-04
6,Ramsay MacDonald,Labour,1929-06-05,1931-08-24
7,Ramsay MacDonald,National Labour,1931-08-24,1935-06-07
8,Stanley Baldwin,Conservative,1935-06-07,1937-05-28
9,Neville Chamberlain,Conservative,1937-05-28,1940-05-10


None


In [55]:
print(election_df[['election_date', 'con_seats', 'lab_seats', 'lib_seats', 'oth_seats', 'total_seats', 'comment']].to_string())

   election_date  con_seats  lab_seats  lib_seats  oth_seats  total_seats                                                           comment
0     1910-12-19        272         42        272         84          670   Repeat election forced by George V to legitimise Parliament Act
1     1918-12-14        383         73        161         90          707                       Lloyd George (Lib) leads coalition with Con
2     1922-11-15        345        142        116         12          615        Bonar Law (Con) breaks the coalition, Labour overtakes Lib
3     1923-12-06        258        191        159          7          615     Ramsay MacDonald (Lab) with Lib-coalition beats Baldwin (Con)
4     1924-10-29        419        151         40          5          615       Baldwin (Con) elected after Liberals bring down Labour govt
5     1929-05-30        260        288         59          8          615             Ramsay MacDonald (Lab) minority govt with Lib support
6     1931-10-27    

In [58]:
# Add all missing columns
for col in ['election_date', 'own_mandate', 'seats_won', 'total_seats', 
            'majority', 'majority_end', 'opposition_party', 'coalition_parties', 
            'resignation_date', 'end_reason', 'resignation', 'notes']:
    df_timeline[col] = None

election_df['election_date'] = pd.to_datetime(election_df['election_date']).dt.strftime('%Y-%m-%d')


# Then export both
df_timeline.to_excel('pm_timeline_working.xlsx', index=False)
election_df.to_excel('election_data.xlsx', index=False)

In [59]:
print(df_timeline[df_timeline['pm'] == 'Ramsay MacDonald'].to_string())

                 pm            party pm_start_date pm_end_date election_date own_mandate seats_won total_seats majority majority_end opposition_party coalition_parties resignation_date end_reason resignation notes
4  Ramsay MacDonald           Labour    1924-01-22  1924-11-04          None        None      None        None     None         None             None              None             None       None        None  None
6  Ramsay MacDonald           Labour    1929-06-05  1931-08-24          None        None      None        None     None         None             None              None             None       None        None  None
7  Ramsay MacDonald  National Labour    1931-08-24  1935-06-07          None        None      None        None     None         None             None              None             None       None        None  None


## Collected Data and Filled in the table in excel

## Reading the table back into pandas

In [62]:
df_timeline = pd.read_excel('pm_timeline_working.xlsx')
print(df_timeline.columns.tolist())
print(df_timeline.shape)
display(df_timeline)

['pm', 'party', 'pm_start_date', 'pm_end_date', 'election_date', 'own_mandate', 'seats_won', 'total_seats', 'opposition_party', 'majority', 'coalition_parties', 'end_reason', 'notes']
(48, 13)


,pm,party,pm_start_date,pm_end_date,election_date,own_mandate,seats_won,total_seats,opposition_party,majority,coalition_parties,end_reason,notes
0,H. H. Asquith,Liberal,1910-12-19,1916-12-06,1910-12-19,True,NaN,NaN,Conservative,NaN,NaN,NaN,NaN
1,David Lloyd George,Coalition Liberal,1916-12-06,1918-12-14,1910-12-19,False,NaN,NaN,Liberal,NaN,"Conservative, Labour",NaN,NaN
2,David Lloyd George,Coalition Liberal,1918-12-14,1922-10-19,1918-12-14,True,521.0,NaN,Liberal,NaN,"Conservative, National Liberal",NaN,NaN
3,Bonar Law,Conservative,1922-10-23,1922-11-15,1918-12-14,False,NaN,NaN,Labour,NaN,NaN,NaN,NaN
4,Bonar Law,Conservative,1922-11-15,1923-05-20,1922-11-15,True,NaN,NaN,Labour,NaN,NaN,NaN,NaN
5,Stanley Baldwin,Conservative,1923-05-22,1923-12-06,1922-11-15,False,NaN,NaN,Labour,NaN,NaN,NaN,NaN
6,Stanley Baldwin,Conservative,1923-12-06,1924-01-22,1923-12-06,True,NaN,NaN,Labour,NaN,NaN,NaN,NaN
7,Ramsay MacDonald,Labour,1924-01-22,1924-11-04,1923-12-06,False,NaN,NaN,Conservative,NaN,NaN,NaN,NaN
8,Stanley Baldwin,Conservative,1924-11-04,1929-06-04,1924-10-29,True,NaN,NaN,Labour,NaN,NaN,NaN,NaN
9,Ramsay MacDonald,Labour,1929-06-05,1931-08-24 00:00:00,1929-05-30,True,NaN,NaN,Conservative,NaN,NaN,NaN,NaN


In [ ]:
for col in ['pm_start_date', 'pm_end_date', 'election_date']:
    df_timeline[col] = pd.to_datetime(df_timeline[col]).dt.strftime('%Y-%m-%d')
display(df_timeline)


In [65]:
# Strip times from election_df election_date first
election_df['election_date'] = pd.to_datetime(election_df['election_date']).dt.strftime('%Y-%m-%d')

# Fill in total_seats from election_df where it's missing
df_timeline['total_seats'] = df_timeline['total_seats'].fillna(
    df_timeline['election_date'].map(election_df.set_index('election_date')['total_seats'])
)

print(df_timeline[['pm', 'election_date', 'total_seats']].to_string())
display(df_timeline)


                     pm election_date  total_seats
0         H. H. Asquith    1910-12-19        670.0
1    David Lloyd George    1910-12-19        670.0
2    David Lloyd George    1918-12-14        707.0
3             Bonar Law    1918-12-14        707.0
4             Bonar Law    1922-11-15        615.0
5       Stanley Baldwin    1922-11-15        615.0
6       Stanley Baldwin    1923-12-06        615.0
7      Ramsay MacDonald    1923-12-06        615.0
8       Stanley Baldwin    1924-10-29        615.0
9      Ramsay MacDonald    1929-05-30        615.0
10     Ramsay MacDonald    1929-05-30        615.0
11     Ramsay MacDonald    1931-10-27        615.0
12      Stanley Baldwin    1931-10-27        615.0
13      Stanley Baldwin    1935-11-14        615.0
14  Neville Chamberlain    1935-11-14        615.0
15    Winston Churchill    1935-11-14        615.0
16       Clement Attlee    1945-07-05        640.0
17       Clement Attlee    1950-02-23        625.0
18    Winston Churchill    1951

,pm,party,pm_start_date,pm_end_date,election_date,own_mandate,seats_won,total_seats,opposition_party,majority,coalition_parties,end_reason,notes
0,H. H. Asquith,Liberal,1910-12-19,1916-12-06,1910-12-19,True,NaN,670.0,Conservative,NaN,NaN,NaN,NaN
1,David Lloyd George,Coalition Liberal,1916-12-06,1918-12-14,1910-12-19,False,NaN,670.0,Liberal,NaN,"Conservative, Labour",NaN,NaN
2,David Lloyd George,Coalition Liberal,1918-12-14,1922-10-19,1918-12-14,True,521.0,707.0,Liberal,NaN,"Conservative, National Liberal",NaN,NaN
3,Bonar Law,Conservative,1922-10-23,1922-11-15,1918-12-14,False,NaN,707.0,Labour,NaN,NaN,NaN,NaN
4,Bonar Law,Conservative,1922-11-15,1923-05-20,1922-11-15,True,NaN,615.0,Labour,NaN,NaN,NaN,NaN
5,Stanley Baldwin,Conservative,1923-05-22,1923-12-06,1922-11-15,False,NaN,615.0,Labour,NaN,NaN,NaN,NaN
6,Stanley Baldwin,Conservative,1923-12-06,1924-01-22,1923-12-06,True,NaN,615.0,Labour,NaN,NaN,NaN,NaN
7,Ramsay MacDonald,Labour,1924-01-22,1924-11-04,1923-12-06,False,NaN,615.0,Conservative,NaN,NaN,NaN,NaN
8,Stanley Baldwin,Conservative,1924-11-04,1929-06-04,1924-10-29,True,NaN,615.0,Labour,NaN,NaN,NaN,NaN
9,Ramsay MacDonald,Labour,1929-06-05,1931-08-24,1929-05-30,True,NaN,615.0,Conservative,NaN,NaN,NaN,NaN


In [ ]:
# Create a mapping of election_date to seats by party
con_seats = election_df.set_index('election_date')['con_seats']
lab_seats = election_df.set_index('election_date')['lab_seats']

# Fill in seats_won where missing based on party
def fill_seats(row):
    if pd.notna(row['seats_won']):
        return row['seats_won']
    if row['party'] == 'Conservative':
        return con_seats.get(row['election_date'])
    elif row['party'] == 'Labour':
        return lab_seats.get(row['election_date'])
    return None

df_timeline['seats_won'] = df_timeline.apply(fill_seats, axis=1)

# print(df_timeline[['pm', 'party', 'election_date', 'seats_won']].to_string())
display(df_timeline)

                     pm              party election_date  seats_won
0         H. H. Asquith            Liberal    1910-12-19        NaN
1    David Lloyd George  Coalition Liberal    1910-12-19        NaN
2    David Lloyd George  Coalition Liberal    1918-12-14      521.0
3             Bonar Law       Conservative    1918-12-14      383.0
4             Bonar Law       Conservative    1922-11-15      345.0
5       Stanley Baldwin       Conservative    1922-11-15      345.0
6       Stanley Baldwin       Conservative    1923-12-06      258.0
7      Ramsay MacDonald             Labour    1923-12-06      191.0
8       Stanley Baldwin       Conservative    1924-10-29      419.0
9      Ramsay MacDonald             Labour    1929-05-30      288.0
10     Ramsay MacDonald    National Labour    1929-05-30        NaN
11     Ramsay MacDonald    National Labour    1931-10-27        NaN
12      Stanley Baldwin       Conservative    1931-10-27      521.0
13      Stanley Baldwin       Conservative    19

,pm,party,pm_start_date,pm_end_date,election_date,own_mandate,seats_won,total_seats,opposition_party,majority,coalition_parties,end_reason,notes
0,H. H. Asquith,Liberal,1910-12-19,1916-12-06,1910-12-19,True,NaN,670.0,Conservative,NaN,NaN,NaN,NaN
1,David Lloyd George,Coalition Liberal,1916-12-06,1918-12-14,1910-12-19,False,NaN,670.0,Liberal,NaN,"Conservative, Labour",NaN,NaN
2,David Lloyd George,Coalition Liberal,1918-12-14,1922-10-19,1918-12-14,True,521.0,707.0,Liberal,NaN,"Conservative, National Liberal",NaN,NaN
3,Bonar Law,Conservative,1922-10-23,1922-11-15,1918-12-14,False,383.0,707.0,Labour,NaN,NaN,NaN,NaN
4,Bonar Law,Conservative,1922-11-15,1923-05-20,1922-11-15,True,345.0,615.0,Labour,NaN,NaN,NaN,NaN
5,Stanley Baldwin,Conservative,1923-05-22,1923-12-06,1922-11-15,False,345.0,615.0,Labour,NaN,NaN,NaN,NaN
6,Stanley Baldwin,Conservative,1923-12-06,1924-01-22,1923-12-06,True,258.0,615.0,Labour,NaN,NaN,NaN,NaN
7,Ramsay MacDonald,Labour,1924-01-22,1924-11-04,1923-12-06,False,191.0,615.0,Conservative,NaN,NaN,NaN,NaN
8,Stanley Baldwin,Conservative,1924-11-04,1929-06-04,1924-10-29,True,419.0,615.0,Labour,NaN,NaN,NaN,NaN
9,Ramsay MacDonald,Labour,1929-06-05,1931-08-24,1929-05-30,True,288.0,615.0,Conservative,NaN,NaN,NaN,NaN


In [67]:
# Asquith and Lloyd George pre-1918 - Liberal seats 1910
df_timeline.loc[0, 'seats_won'] = 274
df_timeline.loc[1, 'seats_won'] = 274

# MacDonald National Labour pre-election - inherited 1929 Labour seats
df_timeline.loc[10, 'seats_won'] = 288

# MacDonald National Labour post-election 1931 - full National Government coalition
df_timeline.loc[11, 'seats_won'] = 554

In [69]:
def calc_majority(row):
    if pd.notna(row['seats_won']) and pd.notna(row['total_seats']):
        return row['seats_won'] - (row['total_seats'] / 2 + 1)
    return None

df_timeline['majority'] = df_timeline.apply(calc_majority, axis=1)

display(df_timeline)

,pm,party,pm_start_date,pm_end_date,election_date,own_mandate,seats_won,total_seats,opposition_party,majority,coalition_parties,end_reason,notes
0,H. H. Asquith,Liberal,1910-12-19,1916-12-06,1910-12-19,True,274.0,670.0,Conservative,-62.0,NaN,NaN,NaN
1,David Lloyd George,Coalition Liberal,1916-12-06,1918-12-14,1910-12-19,False,274.0,670.0,Liberal,-62.0,"Conservative, Labour",NaN,NaN
2,David Lloyd George,Coalition Liberal,1918-12-14,1922-10-19,1918-12-14,True,521.0,707.0,Liberal,166.5,"Conservative, National Liberal",NaN,NaN
3,Bonar Law,Conservative,1922-10-23,1922-11-15,1918-12-14,False,383.0,707.0,Labour,28.5,NaN,NaN,NaN
4,Bonar Law,Conservative,1922-11-15,1923-05-20,1922-11-15,True,345.0,615.0,Labour,36.5,NaN,NaN,NaN
5,Stanley Baldwin,Conservative,1923-05-22,1923-12-06,1922-11-15,False,345.0,615.0,Labour,36.5,NaN,NaN,NaN
6,Stanley Baldwin,Conservative,1923-12-06,1924-01-22,1923-12-06,True,258.0,615.0,Labour,-50.5,NaN,NaN,NaN
7,Ramsay MacDonald,Labour,1924-01-22,1924-11-04,1923-12-06,False,191.0,615.0,Conservative,-117.5,NaN,NaN,NaN
8,Stanley Baldwin,Conservative,1924-11-04,1929-06-04,1924-10-29,True,419.0,615.0,Labour,110.5,NaN,NaN,NaN
9,Ramsay MacDonald,Labour,1929-06-05,1931-08-24,1929-05-30,True,288.0,615.0,Conservative,-20.5,NaN,NaN,NaN


In [70]:
print(df_timeline['own_mandate'].dtype)
print(df_timeline['own_mandate'].unique())

bool
[ True False]


In [71]:
df_timeline.to_json('government_timeline.json', orient='records', indent=4)

In [73]:
print(df)

   election_date  total_seats  con_votes  con_seats  lab_votes  lab_seats  \
0     1910-12-19          670       46.3        272        7.2         42   
1     1918-12-14          707       38.7        383       23.7         73   
2     1922-11-15          615       38.2        345       29.5        142   
3     1923-12-06          615       38.1        258       30.5        191   
4     1924-10-29          615       48.3        419       33.0        151   
5     1929-05-30          615       38.2        260       37.1        288   
6     1931-10-27          615       60.6        521       30.6         52   
7     1935-11-14          615       53.7        432       37.9        154   
8     1945-07-05          640       39.8        213       47.8        393   
9     1950-02-23          625       43.5        298       46.1        315   
10    1951-10-25          625       48.0        321       48.7        295   
11    1955-05-26          630       49.7        345       46.2        277   

In [74]:
import json, re
from pathlib import Path

timeline = json.loads(Path("government_timeline.json").read_text())
raw      = Path("text.txt").read_text()
blocks   = [b.strip() for b in re.split(r'\n\s*\n', raw) if b.strip()]

# Build lookup keyed on (pm_name, start_date)
lookup = {}
for block in blocks:
    hm = re.match(r'^(.+?)\s+\((\d{4}-\d{2}-\d{2})\s+to\s+', block)
    if not hm:
        continue
    rm = re.search(r'^End Reason:\s*(.+)$', block, re.MULTILINE)
    nm = re.search(r'^Notes:\s*(.+)$',      block, re.MULTILINE)
    end_reason = rm.group(1).strip() if rm else None
    if end_reason and end_reason.upper() in ("N/A", "NA"):
        end_reason = None
    lookup[(hm.group(1).strip(), hm.group(2))] = {
        "end_reason": end_reason,
        "notes":      nm.group(1).strip() if nm else None
    }

for row in timeline:
    entry = lookup.get((row["pm"], row["pm_start_date"]))
    if entry:
        row.update(entry)

Path("government_timeline.json").write_text(json.dumps(timeline, indent=4))
print(f"Done. {sum(1 for r in timeline if r.get('end_reason'))}/{len(timeline)} filled.")

Done. 47/48 filled.
